# 03. 상품 공간(place) 라벨 매핑

**목표**: 집들이 이미지의 `place` 컬럼(숫자값)이 무엇을 의미하는지 API로 확인하고 한글 공간명으로 매핑

**흐름**

1. 셀 1 — 전체 데이터 & 게시글 샘플 로드
2. 셀 2 — API 호출 & place 추출 함수 정의
3. 셀 3 — 100개 샘플 게시글에서 place.value ↔ place.label 수집
4. 셀 4 — value별 label 확인 & 충돌 체크
5. 셀 5 — 매핑 딕셔너리에 없는 place_value 10 API로 직접 확인
6. 셀 6 — 전체 데이터에 place 매핑 적용 & CSV 저장

**입력 파일**: `housewarming_posts_final.csv`, `housewarming_image_products_category_final_reviewed.csv`  
**출력 파일**: `housewarming_image_products_category_place_final.csv`

## 셀 1. 전체 데이터 & 게시글 샘플 로드

- `df`: place 매핑 대상 전체 데이터
- `sample_ids`: place.value ↔ place.label 확인용 100개 샘플

In [ ]:
import pandas as pd
import requests
import random
import time
from tqdm import tqdm

# 매핑 대상 전체 데이터
df = pd.read_csv(
    "housewarming_image_products_category_final_reviewed.csv",
    encoding="utf-8-sig"
)
df["place_value"] = pd.to_numeric(df["place"], errors="coerce")
print("df shape:", df.shape)

# place.value ↔ label 확인용 게시글 샘플
posts = pd.read_csv("housewarming_posts_final.csv", encoding="utf-8-sig")
content_ids = (
    posts["contentId"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)
print("전체 게시글 수:", len(content_ids))

random.seed(42)
sample_ids = random.sample(content_ids, min(100, len(content_ids)))
print("샘플 게시글 수:", len(sample_ids))

## 셀 2. API 호출 & place 추출 함수 정의

- `fetch_project_api`: 게시글 API 호출
- `extract_place_label_rows`: API 응답에서 place.value ↔ place.label 추출

In [ ]:
session = requests.Session()

headers = {
    "accept": "application/json",
    "content-type": "application/json",
    "referer": "https://contents.ohou.se/",
    "ohouse-trust-client": "ohs-web-client",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
}


def fetch_project_api(content_id):
    url = f"https://contents.ohou.se/api/projects/{content_id}"
    try:
        res = session.get(url, headers=headers, timeout=20)
        if res.status_code != 200:
            print(f"실패: contentId={content_id}, status={res.status_code}")
            return None
        return res.json()
    except Exception as e:
        print(f"에러: contentId={content_id}, error={e}")
        return None


def extract_place_label_rows(data, content_id):
    """API 응답 전체에서 place.value ↔ place.label 추출"""
    rows = []

    def walk(obj):
        if isinstance(obj, dict):
            if obj.get("type") == "CardImage" and isinstance(obj.get("place"), dict):
                place = obj.get("place", {}) or {}
                rows.append({
                    "contentId":   content_id,
                    "cardId":      obj.get("id"),
                    "place_label": place.get("label"),
                    "place_value": place.get("value"),
                })
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(data)
    return rows


def extract_place_objects(data, content_id):
    """place dict가 있는 모든 객체에서 place.value ↔ place.label 추출 (place_value 10 확인용)"""
    rows = []

    def walk(obj):
        if isinstance(obj, dict):
            if isinstance(obj.get("place"), dict):
                place = obj["place"]
                rows.append({
                    "contentId":   content_id,
                    "cardId":      obj.get("id"),
                    "place_value": place.get("value"),
                    "place_label": place.get("label")
                })
            for v in obj.values():
                walk(v)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(data)
    return rows


print("함수 정의 완료")

## 셀 3. 100개 샘플에서 place.value ↔ place.label 수집

API 응답에서 place.value(숫자)와 place.label(한글명)을 함께 수집해  
어떤 숫자가 어떤 공간을 뜻하는지 확인합니다.

In [ ]:
place_label_rows = []

for content_id in tqdm(sample_ids):
    data = fetch_project_api(content_id)
    if data is None:
        continue
    place_label_rows.extend(extract_place_label_rows(data, content_id))
    time.sleep(0.2)

place_label_df = pd.DataFrame(place_label_rows)

print("수집된 place rows:", place_label_df.shape)
place_label_df.head()

## 셀 4. place.value별 label 확인 & 충돌 체크

같은 value에 여러 label이 붙은 경우가 있으면 매핑 딕셔너리 작성 전에 확인이 필요합니다.

In [ ]:
value_label_check = (
    place_label_df
    .dropna(subset=["place_value", "place_label"])
    .groupby("place_value")["place_label"]
    .unique()
    .reset_index()
)
value_label_check["label_count"] = value_label_check["place_label"].apply(len)
value_label_check = value_label_check.sort_values("place_value")

print("place_value별 label 목록:")
display(value_label_check)

# 같은 value에 label이 2개 이상인 충돌 확인
conflict = value_label_check[value_label_check["label_count"] >= 2]
print("\n충돌(같은 value에 여러 label):", len(conflict), "건")
if len(conflict) > 0:
    display(conflict)

## 셀 5. 매핑되지 않은 place_value 10 직접 확인

셀 4에서 샘플에 없던 place_value 10을 API로 직접 조회해 label을 확인합니다.  
→ **상업공간** 으로 확인됨

In [ ]:
# df에서 place_value 10인 게시글 샘플 추출
place10_content_ids = (
    df[df["place_value"] == 10]["contentId"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .head(10)
    .tolist()
)
print("place_value 10 게시글 샘플:", place10_content_ids)

# API로 place.label 직접 확인
place10_label_rows = []
for content_id in tqdm(place10_content_ids):
    data = fetch_project_api(content_id)
    if data is None:
        continue
    place10_label_rows.extend(extract_place_objects(data, content_id))
    time.sleep(0.2)

place10_label_df = pd.DataFrame(place10_label_rows)

print("\nplace_value 10의 label 확인:")
display(
    place10_label_df[place10_label_df["place_value"] == 10]
    .drop_duplicates()
)

## 셀 6. 전체 데이터에 place 매핑 적용 & CSV 저장

셀 3~5에서 확인된 내용을 바탕으로 매핑 딕셔너리를 완성하고 전체 데이터에 적용합니다.

In [ ]:
# =========================================================
# place_value → 한글 공간명 매핑 딕셔너리
# (셀 3~5 API 탐색으로 확인된 값 기준)
# =========================================================
place_label_map = {
     0: "원룸",
     1: "거실",
     2: "침실",
     3: "주방",
     4: "욕실",
     5: "아이방",
     6: "드레스룸",
     7: "서재&작업실",
     8: "베란다",
     9: "사무공간",
    10: "상업공간",   # 셀 5에서 API 직접 확인
    11: "가구&소품",
    12: "현관",
    13: "외관&기타",
    14: "제품후기",
    15: "자기소개",
    16: "비포사진",
    17: "복도",
    18: "전문가리뷰",
    19: "전문가소개",
    20: "도면 및 가구배치도",
    21: "시공 리뷰"
}

# 매핑 적용
df["place_label"] = df["place_value"].map(place_label_map)

# 미매핑 확인
unmapped = df[df["place_value"].notna() & df["place_label"].isna()]
print("미매핑 place 값 수:", unmapped["place_value"].nunique())
if unmapped["place_value"].nunique() > 0:
    print(unmapped["place_value"].value_counts().sort_index())
else:
    print("모든 place_value 정상 매핑됨")

print("\nplace_label 분포:")
print(df["place_label"].value_counts(dropna=False))

# 저장
output_file = "housewarming_image_products_category_place_final.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\n저장 완료:", output_file)
print("전체 행 수:", len(df))